In [22]:
DEBUG = False

TRAIN_SIZE = 500 if DEBUG else 5000
QA_SELECTION_SIZE = 200 if DEBUG else 300

RETRIEVAL_DATASET = "/kaggle/input/datasets/keyaaness/cost-aware-adaptive-rag-retrieval-v1/retrieval_artifacts"
OUTPUT_DIR = "/kaggle/working/controller_artifacts_v2_final"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 64
SEED = 42

!pip install -q sentence-transformers transformers accelerate xgboost pandas numpy pyarrow faiss-cpu scikit-learn

import os
import re
import json
import pickle
import random
import time

import numpy as np
import pandas as pd
import torch
import faiss

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

EXPANSION_RATES = np.arange(0.10, 0.51, 0.05)


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

corpus = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "corpus.parquet")
)

controller_train = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "controller_train.parquet")
)

controller_val = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "controller_validation.parquet")
)

controller_train = controller_train.sample(
    n=TRAIN_SIZE,
    random_state=SEED
).reset_index(drop=True)

qa_selection = controller_val.sample(
    n=QA_SELECTION_SIZE,
    random_state=SEED
).reset_index(drop=True)

faiss_index = faiss.read_index(
    os.path.join(RETRIEVAL_DATASET, "faiss.index")
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    use_fast=True
)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

generator = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=dtype
)

generator.to(device)
generator.eval()

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token = generator_tokenizer.eos_token

print("Device:", device)
print("Training examples:", len(controller_train))
print("QA selection examples:", len(qa_selection))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Device: cuda
Training examples: 5000
QA selection examples: 300


In [23]:
FEATURE_NAMES = [
    "query_tokens",
    "query_chars",
    "top1",
    "top2",
    "top3",
    "gap12",
    "gap23",
    "gap13",
    "mean3",
    "std3",
    "top1_mean3_ratio"
]

def make_retrieval_features(
    questions,
    dataframe
):
    embeddings = embedding_model.encode(
        questions,
        batch_size=128,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        device=device
    ).astype(np.float32)

    scores, indices = faiss_index.search(
        embeddings,
        5
    )

    X = []
    y = []

    for i, row in dataframe.iterrows():
        s = scores[i]

        support_ids = set(
            json.loads(
                row["supporting_doc_ids"]
            )
        )

        retrieved3 = set(
            int(x)
            for x in indices[i][:3]
        )

        retrieved5 = set(
            int(x)
            for x in indices[i][:5]
        )

        recall3 = (
            len(retrieved3 & support_ids)
            / max(len(support_ids), 1)
        )

        recall5 = (
            len(retrieved5 & support_ids)
            / max(len(support_ids), 1)
        )

        X.append({
            "query_tokens": len(
                re.findall(
                    r"[A-Za-z0-9]+",
                    str(row["question"]).lower()
                )
            ),
            "query_chars": len(str(row["question"])),
            "top1": float(s[0]),
            "top2": float(s[1]),
            "top3": float(s[2]),
            "gap12": float(s[0] - s[1]),
            "gap23": float(s[1] - s[2]),
            "gap13": float(s[0] - s[2]),
            "mean3": float(s[:3].mean()),
            "std3": float(s[:3].std()),
            "top1_mean3_ratio": float(
                s[0] / (abs(s[:3].mean()) + 1e-8)
            )
        })

        y.append(
            int(recall5 > recall3)
        )

    return (
        pd.DataFrame(
            X,
            columns=FEATURE_NAMES
        ).astype(np.float32),
        np.asarray(y, dtype=np.int32),
        embeddings,
        indices,
        scores
    )

X_train, y_train, _, _, _ = make_retrieval_features(
    controller_train["question"].astype(str).tolist(),
    controller_train
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

positive = max(
    int(y_train.sum()),
    1
)

negative = max(
    len(y_train) - positive,
    1
)

controller = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=negative / positive,
    random_state=SEED,
    n_jobs=2
)

controller.fit(
    X_train_scaled,
    y_train
)

print("Positive train labels:", int(y_train.sum()))
print("Negative train labels:", int((y_train == 0).sum()))

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Positive train labels: 508
Negative train labels: 4492


In [24]:
def build_context(documents):
    return "\n\n".join(
        f"[Document {i}]\n"
        f"Title: {d['title']}\n"
        f"{d['text']}"
        for i, d in enumerate(documents, 1)
    )


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def exact_match(prediction, reference):
    return float(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1(prediction, reference):
    p = normalize_answer(prediction).split()
    r = normalize_answer(reference).split()

    if not p and not r:
        return 1.0

    if not p or not r:
        return 0.0

    pc = {}
    rc = {}

    for token in p:
        pc[token] = pc.get(token, 0) + 1

    for token in r:
        rc[token] = rc.get(token, 0) + 1

    overlap = sum(
        min(pc[token], rc.get(token, 0))
        for token in pc
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(r)

    return (
        2 * precision * recall
        / (precision + recall)
    )


def retrieve_from_scores(
    row_indices,
    k
):
    documents = []

    for idx in row_indices[:k]:
        row = corpus.iloc[int(idx)]

        documents.append({
            "doc_id": int(row["doc_id"]),
            "title": str(row["title"]),
            "text": str(row["text"])
        })

    return documents


def generate_answer(
    question,
    context
):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the provided context. "
                "Return only the concise answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}\n"
                f"Answer:"
            )
        }
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = generator.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=generator_tokenizer.pad_token_id,
            eos_token_id=generator_tokenizer.eos_token_id
        )

    return generator_tokenizer.decode(
        output[0][prompt_length:],
        skip_special_tokens=True
    ).strip()


X_val, _, _, val_indices, val_scores = make_retrieval_features(
    qa_selection["question"].astype(str).tolist(),
    qa_selection
)

router_scores = controller.predict_proba(
    scaler.transform(X_val)
)[:, 1]

f1_at_3 = []
f1_at_5 = []

for i, row in tqdm(
    qa_selection.iterrows(),
    total=len(qa_selection),
    desc="Validation QA"
):
    documents3 = retrieve_from_scores(
        val_indices[i],
        3
    )

    documents5 = retrieve_from_scores(
        val_indices[i],
        5
    )

    answer3 = generate_answer(
        row["question"],
        build_context(documents3)
    )

    answer5 = generate_answer(
        row["question"],
        build_context(documents5)
    )

    f1_at_3.append(
        token_f1(
            answer3,
            row["answer"]
        )
    )

    f1_at_5.append(
        token_f1(
            answer5,
            row["answer"]
        )
    )

f1_at_3 = np.asarray(
    f1_at_3,
    dtype=np.float32
)

f1_at_5 = np.asarray(
    f1_at_5,
    dtype=np.float32
)

operating_points = []

order = np.argsort(
    router_scores
)[::-1]

for rate in EXPANSION_RATES:
    n_expand = max(
        1,
        int(
            np.ceil(
                rate * len(qa_selection)
            )
        )
    )

    expanded = np.zeros(
        len(qa_selection),
        dtype=bool
    )

    expanded[
        order[:n_expand]
    ] = True

    adaptive_f1 = float(
        np.where(
            expanded,
            f1_at_5,
            f1_at_3
        ).mean()
    )

    avg_k = float(
        np.where(
            expanded,
            5,
            3
        ).mean()
    )

    operating_points.append({
        "expansion_rate": float(rate),
        "avg_k": avg_k,
        "adaptive_f1": adaptive_f1,
        "f1_at_3": float(f1_at_3.mean()),
        "f1_at_5": float(f1_at_5.mean())
    })

operating_df = pd.DataFrame(
    operating_points
)

best_f1 = operating_df["adaptive_f1"].max()

eligible = operating_df[
    operating_df["adaptive_f1"]
    >= best_f1 - 0.005
].copy()

best_idx = eligible["avg_k"].idxmin()

selected_rate = float(
    operating_df.loc[
        best_idx,
        "expansion_rate"
    ]
)

print(
    operating_df.to_string(
        index=False
    )
)

print(
    "\nSelected expansion rate:",
    selected_rate
)

print(
    "Selected validation F1:",
    operating_df.loc[
        best_idx,
        "adaptive_f1"
    ]
)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Validation QA:   0%|          | 0/300 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 expansion_rate    avg_k  adaptive_f1  f1_at_3  f1_at_5
           0.10 3.200000     0.470328 0.471998 0.465977
           0.15 3.306667     0.470668 0.471998 0.465977
           0.20 3.406667     0.467564 0.471998 0.465977
           0.25 3.506667     0.467564 0.471998 0.465977
           0.30 3.606667     0.467564 0.471998 0.465977
           0.35 3.706667     0.468675 0.471998 0.465977
           0.40 3.806667     0.471308 0.471998 0.465977
           0.45 3.906667     0.467974 0.471998 0.465977
           0.50 4.006667     0.471170 0.471998 0.465977

Selected expansion rate: 0.1
Selected validation F1: 0.470327764749527


In [25]:
output_path = Path(
    OUTPUT_DIR
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    output_path / "controller.pkl",
    "wb"
) as f:
    pickle.dump(
        controller,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    output_path / "scaler.pkl",
    "wb"
) as f:
    pickle.dump(
        scaler,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    output_path / "feature_names.json",
    "w"
) as f:
    json.dump(
        FEATURE_NAMES,
        f,
        indent=2
    )

with open(
    output_path / "selected_expansion_rate.json",
    "w"
) as f:
    json.dump(
        {
            "expansion_rate": selected_rate,
            "base_k": 3,
            "expanded_k": 5,
            "selection_rule": (
                "Best validation F1; among F1 values "
                "within 0.005 of best, select lowest average k."
            )
        },
        f,
        indent=2
    )

operating_df.to_csv(
    output_path /
    "validation_operating_points.csv",
    index=False
)

importance = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "importance": controller.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance.to_csv(
    output_path /
    "feature_importance.csv",
    index=False
)

summary = {
    "train_examples": int(len(controller_train)),
    "qa_selection_examples": int(len(qa_selection)),
    "train_positive_rate": float(y_train.mean()),
    "selected_expansion_rate": selected_rate,
    "selected_average_k": float(
        operating_df.loc[
            best_idx,
            "avg_k"
        ]
    ),
    "selected_validation_f1": float(
        operating_df.loc[
            best_idx,
            "adaptive_f1"
        ]
    ),
    "fixed_top3_validation_f1": float(
        f1_at_3.mean()
    ),
    "fixed_top5_validation_f1": float(
        f1_at_5.mean()
    )
}

with open(
    output_path /
    "controller_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    pd.Series(summary).to_string()
)

train_examples              5000.000000
qa_selection_examples        300.000000
train_positive_rate            0.101600
selected_expansion_rate        0.100000
selected_average_k             3.200000
selected_validation_f1         0.470328
fixed_top3_validation_f1       0.471998
fixed_top5_validation_f1       0.465977
